# **Cartes extraites du Malaria Atlas Project (MAP)**

## Indicateurs issus du MAP

Ce rapport génère des visualisations géographiques d'indicateurs liés au paludisme, extraits du Malaria Atlas Project (MAP). 

Le rapport se base sur les valeurs des paramètres utilisées dans l'extraction depuis MAP:
- `YEAR_END` : l'année de référence pour les données affichées sur les cartes ; il représente aussi l'année de fin de l'extraction MAP, donc l'année la plus récente pour laquelle les données MAP ont été extraites.

**Note**
- Le paramètre`YEAR_START`, qui représente l'année de début pour les données affichées sur les cartes, n'est pas utilisé dans ce rapport, puisque les cartes sont réalisées uniquement pour l'année la plus récente

Les cartes choroplèthes générées présentent chacune des indicateurs de santé au niveau administratif ADM2, à l'échelle nationale, pour l'année la plus récente de l'extraction, permettant ainsi une analyse visuelle des variations géographiques des indicateurs relatifs au paludisme. Les indicateurs extraits et visualisés sont:

- **`Pf_Parasite_Rate`** : Taux de parasitémie à _Plasmodium falciparum_ (Proportion d'enfants âgés de 2 à 10 ans présentant, au cours d'une année donnée, une parasitémie détectable à _Plasmodium falciparum_)
- **`Pf_Incidence_Rate`** : Taux d'incidence de _Plasmodium falciparum_ (Nombre de nouveaux cas de _Plasmodium falciparum_ diagnostiqués, pour 1 000 habitants, au cours d'une année donnée)
- **`Pf_Mortality_Rate`** : Taux de mortalité de _Plasmodium falciparum_ (Nombre de décès dus à _Plasmodium falciparum_ pour 100 000 habitants au cours d'une année donnée)
- **`Insecticide_Treated_Net_Access`** : Accès aux moustiquaires imprégnées d'insecticide (Proportion de la population ayant accès à une moustiquaire imprégnée d'insecticide dans son foyer au cours d'une année donnée)
- **`Insecticide_Treated_Net_Use_Rate`** : Taux d'utilisation des moustiquaires imprégnées d'insecticide (Proportion de personnes dormant sous une moustiquaire imprégnée d'insecticide, parmi celles ayant accès à une telle moustiquaire au sein de leur foyer au cours d'une année donnée)
- **`IRS_Coverage`** : Couverture par la pulvérisation intradomiciliaire à effet rémanent (Proportion de ménages bénéficiant de la pulvérisation intradomiciliaire à effet rémanent au cours d'une année donnée)
- **`Antimalarial_Effective_Treatment`** : Traitement antipaludique efficace (Proportion de cas de paludisme bénéficiant d'un traitement efficace par un médicament antipaludique)

## 1. Configuration

In [ ]:
# Project paths
ROOT_PATH <- '~/workspace'
PROJECT_PATH <- file.path(ROOT_PATH, "pipelines/snt_map_extracts")
CODE_PATH <- file.path(ROOT_PATH, 'code')
UTILS_PATH <- file.path(PROJECT_PATH, 'utils')

In [ ]:
source(file.path(UTILS_PATH, "snt_map_extracts_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
setup_var <- get_setup_variables(packages= c("arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "httr", "reticulate", "glue"))
config_json <- load_snt_config(file.path(setup_var$CONFIG_PATH, "SNT_config.json"))

# Save config variables
DATASET_MAP <- config_json$SNT_DATASET_IDENTIFIERS$SNT_MAP_EXTRACTS
DATASET_FORMATTED <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

### Paramètres

Importation des paramètres enregistrés lors de l'extraction de données depuis MAP.

In [ ]:
# Load parameters
map_parameters <- load_dataset_file(dataset_id = DATASET_MAP, filename = paste0(COUNTRY_CODE, "_parameters.json"))
cat(jsonlite::toJSON(map_parameters, pretty = TRUE, auto_unbox = TRUE), "\n")


## 2. Chargement et pré-processing des données à cartographier

Les données utilisées

- données administratives : fond de carte au niveau administratif ADM2
- données extraites de MAP (voir les explications de chaque indicateur en haut du rapport):
    - Pf_Parasite_Rate
    - Pf_Incidence_Rate
    - Pf_Mortality_Rate
    - Insecticide_Treated_Net_Access
    - Insecticide_Treated_Net_Use_Rate
    - IRS_Coverage
    - Antimalarial_Effective_Treatment


In [ ]:

# Load latest year file available
map_data <- load_dataset_file(dataset_id = DATASET_MAP, filename = glue("{COUNTRY_CODE}_map_data_{map_parameters$YEAR_END}.parquet"))

# import DHIS2 shapes data
shapes_data <- load_dataset_file(dataset_id = DATASET_FORMATTED, filename = paste0(COUNTRY_CODE, "_shapes.geojson"))

In [ ]:
print(glue("Année de référence: {map_parameters$YEAR_END}"))

## 3. Génération des graphiques de résultat 

Ces cartes sont générées sur base des données sur l'année la plus récente de l'extraction.

In [ ]:
# Merge geometry with map data
map_data_joined <- dplyr::left_join(shapes_data, map_data, by = c("ADM2_ID" = "ADM2_ID"))

# Get list of metrics
metrics <- unique(map_data$METRIC_NAME)

# Create one map per metric
plots <- build_metric_plots(map_data_joined = map_data_joined, metrics = metrics, year=map_parameters$YEAR_END)

In [ ]:
# Set plot size for individual display
options(repr.plot.width = 10, repr.plot.height = 8)

# Loop through plots and print one by one
for (p in plots) {
  print(p)
  Sys.sleep(1)  # Optional: short pause between plots
}